In [ ]:
%load_ext autoreload
%autoreload 2


import numpy as np
import pandas as pd
import torch
import pydicom
import matplotlib.pyplot as plt

from pathlib import Path

# MONAI imports
import monai
from monai.data import Dataset, CacheDataset, DataLoader, PILReader
from monai.transforms import (
    LoadImage, LoadImaged, Resized, Compose, SaveImage, 
    Spacingd, SpatialCropd, ResizeWithPadOrCropd
)

import numpy as np
from monai.transforms import (
    Compose,
    LoadImaged,
    Transposed,
    NormalizeIntensityd,
    MapTransform,
    ScaleIntensityRangePercentilesd,
    RandAffined, RandGaussianNoised, 
    RandStdShiftIntensityd, RandScaleIntensityd, RandAdjustContrastd, RandHistogramShiftd,
    ScaleIntensityd, Lambdad,
    LoadImage, Transpose
)

from torch.utils.data import DataLoader
from tqdm.notebook import tqdm



import landmarker
import landmarker.datasets
from landmarker.datasets import get_cepha_landmark_datasets
from landmarker.heatmap import GaussianHeatmapGenerator
from landmarker.models import OriginalSpatialConfigurationNet
from landmarker.losses import GaussianHeatmapL2Loss
from torch.utils.data import DataLoader
from landmarker.visualize import inspection_plot
from landmarker.visualize.utils import prediction_inspect_plot, prediction_inspect_plot_transposed, inspection_plot_numbers

from landmarker.heatmap.generator import GaussianHeatmapGenerator

from landmarker.data import LandmarkDataset

#   My stuff
import ra_utils
import ra_utils.data.data_utils
from  ra_utils.data.data_utils import (
    extract_extras_from_filename, 
    extract_extras_from_abspath
)

import ra_utils.visualization.plot_landmarks
import ra_utils.data
import ra_utils.data.data_handler
import ra_utils.data.dataloader_CR_landmarks
import ra_utils.visualization.plot_landmarks #.plot_landmarks
import pydicom
import numpy as np

import pydicom
import numpy as np
import pandas as pd

from ra_utils.data.data_utils import get_dicom_info



import numpy as np
from sklearn.model_selection import KFold

from ra_utils.data.splits_utils import generate_split_dictionary

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)


In [ ]:
base_dir = Path("/home/cwatzenboeck/data/AutoPIX_cirdata/projects__autoscora/")
# base_dir = Path("/home/clemens/data/AutoPIX_cirdata/")
# base_dir = Path("/home/clemens/data/AutoPIX_cirdata_local/")



dataHandler = ra_utils.data.data_handler.DataHandler_CR_autoscoRA(
                folder_H_images = base_dir / "autoscoRA_images/H_images_of_interest_2_renamed_mirrored_inverted_dicoms",
                folder_F_images = base_dir / "autoscoRA_images/F_images_of_interest_2_renamed_mirrored_inverted_dicoms",
                df_lm_labels_H = base_dir / "landmark_data/100_all_H_joints36/points.csv",
                df_lm_labels_F = base_dir / "landmark_data/100_all_F_joints27/points.csv",
                df_autoscoRA_labels_F = base_dir / "autoscoRA_data/autoscoRA_feet.csv",
                df_autoscoRA_labels_H = base_dir / "autoscoRA_data/autoscoRA_hands.csv",
                training_test_splits_json_H = base_dir / "landmark_data/splits/splits_H_TD_25-03-05_tiny.yml",
                #training_test_splits_json_H = base_dir / "landmark_data/splits/splits_H_TD_25-03-06__exclude_images_without_header.json",
                training_test_splits_json_F = base_dir / "landmark_data/splits/splits_F_TD_25-03-05.yml",
            )



In [ ]:

# import yaml

# def save_dict_to_yaml(data: dict, filename: str):
#     """
#     Saves a Python dictionary to a YAML file in a readable format.
    
#     Parameters:
#     - data (dict): The dictionary to save.
#     - filename (str): The path to the YAML file to save to.
#     """
#     with open(filename, 'w') as file:
#         yaml.dump(data, file, default_flow_style=False, sort_keys=False, allow_unicode=True)


# save_dict_to_yaml(dataHandler.splits_dict_H_raw, "/home/cwatzenboeck/data/AutoPIX_cirdata/projects__autoscora/landmark_data/splits/splits_H_TD_25-03-05.yml")
# save_dict_to_yaml(dataHandler.splits_dict_F_raw, "/home/cwatzenboeck/data/AutoPIX_cirdata/projects__autoscora/landmark_data/splits/splits_F_TD_25-03-05.yml")

In [ ]:
# all_cv = set(dataHandler.splits_dict_H_raw["cv"][0]["test"] +
#              dataHandler.splits_dict_H_raw["cv"][0]["train"])


# all = set((dataHandler.splits_dict_H_raw["training"] +
#            # dataHandler.splits_dict_H_raw["exclude"] +
#            dataHandler.splits_dict_H_raw["test1"] +
#            dataHandler.splits_dict_H_raw["test2"]))


# all_cv == all

# set(dataHandler.splits_dict_H_raw["exclude"]) & set(dataHandler.splits_dict_H_raw["test2"])

In [ ]:
# # Maybe add spacing in better way
df = dataHandler.df_images_and_landmarks_F
# # df["image"]
df_dcm_infos = get_dicom_info(df["image"], only_header=True)


In [ ]:
df_dcm_infos[df_dcm_infos["pixel_spacing"].isna()].reset_index()["file_path"].apply(lambda x: x.stem).to_list()

df_dcm_infos

In [ ]:
# df_dcm_infos.reset_index()[["pixel_spacing_0", "pixel_spacing_1"]]#.mean()

# TODO only eclude the ones which make problems


In [ ]:
# Define transforms 
#from landmarker.transforms.images import UseOnlyFirstChannel

fn_keys = ('image',)
spatial_transformd = [RandAffined(fn_keys, prob=1,
                        rotate_range=(-np.pi/12, np.pi/12),
                        translate_range=(-10, 10),
                        scale_range=(-0.1, 0.1),
                        shear_range=(-0.1, 0.1)
                        )]

train_transformd = Compose([
                            #UseOnlyFirstChannel(('image', )),
                            Transposed(keys=["image"], indices=(0, 2, 1)), # CW added
                            RandGaussianNoised(('image', ), prob=0.2, mean=0, std=1),  # Add gaussian noise
                            RandScaleIntensityd(('image', ), factors=0.25, prob=0.2),  # Add random intensity scaling
                            RandAdjustContrastd(('image', ), prob=0.2, gamma=(0.5,4.5)),  # Randomly adjust contrast
                            RandHistogramShiftd(('image', ), prob=0.2),  # Randomly shift histogram
                            ScaleIntensityd(('image', )),  # Scale intensity
                        ] + spatial_transformd)

# train_transformd = Compose([
#                             #UseOnlyFirstChannel(('image', )),
#                             Transposed(keys=["image"], indices=(0, 2, 1)), # CW added
#                             RandGaussianNoised(('image', ), prob=1, mean=0, std=1.0),  # Add gaussian noise
#                             RandScaleIntensityd(('image', ), factors=1, prob=0.2),  # Add random intensity scaling
#                             RandAdjustContrastd(('image', ), prob=1, gamma=(2.5,2.5)),  # Randomly adjust contrast
#                             RandHistogramShiftd(('image', ), prob=1),  # Randomly shift histogram
#                             ScaleIntensityd(('image', )),  # Scale intensity
#                         ] + spatial_transformd)

inference_transformd = Compose([
    #UseOnlyFirstChannel(('image', )),
    Transposed(keys=["image"], indices=(0, 2, 1)),  # CW added
    ScaleIntensityd(('image', )),
])




In [ ]:

dim_image = (512,512)   # Epoch 50/50 - Train loss: 449.5895 - Val loss: 449.1059 - Val mpe: 4.5445  # Test MPE 3.95
# dim_image = (256, 256)    # Epoch 50/50 - Train loss: 435.0478 - Val loss: 429.6409 - Val mpe: 7.4308    #         6.0
# dim_image = (720,720)  # Epoch 50/50 - Train loss: 452.8256 - Val loss: 452.5538 - Val mpe: 5.3274


(
    image_paths_train,
    image_paths_test1,
    image_paths_test2,
    landmarks_train,
    landmarks_test1,
    landmarks_test2,
    pixel_spacings_train,
    pixel_spacings_test1,
    pixel_spacings_test2
) = dataHandler.get_landmarks_dataset_F(get_pixel_spacing=True)


if not isinstance(pixel_spacings_train, (list, tuple, np.ndarray)) or pixel_spacings_train is None:
    pixel_spacings_train = (0.1, 0.1)
if not isinstance(pixel_spacings_test1, (list, tuple, np.ndarray)) or pixel_spacings_test1 is None:
    pixel_spacings_test1 = (0.1, 0.1)
if not isinstance(pixel_spacings_test2, (list, tuple, np.ndarray)) or pixel_spacings_test2 is None:
    pixel_spacings_test2 = (0.1, 0.1)

N_train = len(image_paths_train)
# TODO Only do not load some of the images

ds_train, ds_test1, ds_test2 = ra_utils.data.dataloader_CR_landmarks.get_landmark_datasets(
    image_paths_train = image_paths_train,
    image_paths_test1 = image_paths_test1,
    image_paths_test2 = image_paths_test2,
    landmarks_train = landmarks_train,
    landmarks_test1 = landmarks_test1,
    landmarks_test2 = landmarks_test2,
    pixel_spacings_train=pixel_spacings_train,
    pixel_spacings_test1=pixel_spacings_test1,
    pixel_spacings_test2=pixel_spacings_test2,
    train_transform=train_transformd,
    inference_transform=inference_transformd,
    dim_img=dim_image
    )

N_landmarks = landmarks_train.shape[1]


In [ ]:


heatmap_generator = GaussianHeatmapGenerator(
    nb_landmarks=N_landmarks,
    sigmas=5.0,
    gamma=10,
    heatmap_size=dim_image,
    learnable=True, # If True, the heatmap generator will be trainable
)



In [ ]:
# Plot the first 3 images from the training set
inspection_plot(ds_train, range(5), heatmap_generator=heatmap_generator)



In [ ]:
# Plot the first 3 images from dataset without transforms
heatmap_generator.device = "cpu" # because dataset tensors are still on cpu
inspection_plot(ds_test1, 0, heatmap_generator=heatmap_generator)
heatmap_generator.device = device # set the desired device back

In [ ]:
# Redefine heatmaps and make learnable: 
heatmap_generator = GaussianHeatmapGenerator(
    nb_landmarks=N_landmarks,
    sigmas=torch.tensor(5.0, device=device),
    gamma=10,
    heatmap_size=dim_image,
    learnable=True, # If True, the heatmap generator will be trainable
)



In [ ]:
N_landmarks

In [ ]:
# Init model, opt, ...
from landmarker.models.spatial_configuration_net import OriginalSpatialConfigurationNet
from landmarker.losses import GaussianHeatmapL2Loss

model = OriginalSpatialConfigurationNet(in_channels=1, out_channels=N_landmarks).to(device)
print("Number of learnable parameters: {}".format(
    sum(p.numel() for p in model.parameters() if p.requires_grad)))
lr = 1e-5
batch_size = 1
epochs =10

# optimizer = torch.optim.SGD([
#     {'params': model.parameters(), "weight_decay":1e-3},
#     {'params': heatmap_generator.sigmas},
#     {'params': heatmap_generator.rotation}]
#     , lr=lr, momentum=0.99, nesterov=True)


optimizer = torch.optim.AdamW(
    [
        {'params': model.parameters(), 'weight_decay': 1e-3},
        {'params': heatmap_generator.sigmas, 'weight_decay': 0.0},
        {'params': heatmap_generator.rotation, 'weight_decay': 0.0},
    ],
    lr=lr
)


criterion = GaussianHeatmapL2Loss(
    alpha=1.0/27
)

lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                                          patience=10, verbose=True, cooldown=10)

next(model.parameters()).is_cuda # returns a boolean

In [ ]:
## Set up dataloaders
train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(ds_test1, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(ds_test2, batch_size=batch_size, shuffle=False, num_workers=0)

In [ ]:
## Train model

from landmarker.heatmap.decoder import heatmap_to_coord
from landmarker.metrics import point_error

def train_epoch(model, heatmap_generator, train_loader, criterion, optimizer, device):
    running_loss = 0
    model.train()
    for i, batch in enumerate(tqdm(train_loader)):
        images = batch["image"].to(device)
        landmarks = batch["landmark"].to(device)
        optimizer.zero_grad()
        outputs = model(images)
        heatmaps = heatmap_generator(landmarks)
        loss = criterion(outputs, heatmap_generator.sigmas, heatmaps)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(train_loader)

def val_epoch(model, heatmap_generator, val_loader, criterion, device, method="local_soft_argmax"):
    eval_loss = 0
    eval_mpe = 0
    model.eval()
    with torch.no_grad():
        for i, batch in enumerate(tqdm(val_loader)):
            images = batch["image"].to(device)
            landmarks = batch["landmark"].to(device)
            outputs = model(images)
            dim_orig = batch["dim_original"].to(device)
            pixel_spacing = batch["spacing"].to(device)
            padding = batch["padding"].to(device)
            heatmaps = heatmap_generator(landmarks)
            loss = criterion(outputs, heatmap_generator.sigmas, heatmaps)
            pred_landmarks = heatmap_to_coord(outputs, method=method)
            eval_loss += loss.item()
            eval_mpe += point_error(landmarks, pred_landmarks, images.shape[-2:], dim_orig,
                                    pixel_spacing, padding, reduction="mean")
    return eval_loss / len(val_loader), eval_mpe / len(val_loader)

def train(model, heatmap_generator, train_loader, val_loader, criterion, optimizer, device, epochs=1000):
    for epoch in tqdm(range(epochs)):
        train_loss = train_epoch(model, heatmap_generator, train_loader, criterion, optimizer, device)
        val_loss, val_mpe = val_epoch(model, heatmap_generator, val_loader, criterion, device)
        print(f"Epoch {epoch+1}/{epochs} - Train loss: {train_loss:.4f} - Val loss: {val_loss:.4f} - Val mpe: {val_mpe:.4f}")
        lr_scheduler.step(val_loss)

In [ ]:
train(model, heatmap_generator, train_loader, val_loader, criterion, optimizer, device,
      epochs=epochs)

### Evaluate model: 

In [ ]:
model.to(device)
next(model.parameters()).is_cuda # returns a boolean

#heatmap_generator.sigmas.shape
heatmap_generator.sigmas

heatmap_generator.gamma

In [ ]:


pred_landmarks = []
true_landmarks = []
dim_origs = []
pixel_spacings = []
paddings = []
test_mpe = 0
model.to(device)
model.eval()
with torch.no_grad():
    for i, X in enumerate(tqdm(test_loader)):
        images = X["image"]
        landmarks = X["landmark"]
        affine_matrix = X["affine"]
        dim_orig = X["dim_original"] 
        pixel_spacing = X["spacing"]
        padding = X["padding"]
        
        images = images.to(device)
        landmarks = landmarks.to(device)
        dim_orig = dim_orig.to(device)
        pixel_spacing = pixel_spacing.to(device)
        padding = padding.to(device)
        outputs = model(images)
        # heatmap = heatmap_generator(landmarks)
        offset_coords = outputs.shape[1]-landmarks.shape[1]
        pred_landmark = heatmap_to_coord(outputs, offset_coords=offset_coords,
                                        method="local_soft_argmax")
        test_mpe += point_error(landmarks, pred_landmark, images.shape[-2:], dim_orig,
                                pixel_spacing, padding, reduction="mean")
        pred_landmarks.append(pred_landmark.cpu())
        true_landmarks.append(landmarks.cpu())
        dim_origs.append(dim_orig.cpu())
        pixel_spacings.append(pixel_spacing.cpu())
        paddings.append(padding.cpu())

pred_landmarks = torch.cat(pred_landmarks)
true_landmarks = torch.cat(true_landmarks)
dim_origs = torch.cat(dim_origs)
pixel_spacings = torch.cat(pixel_spacings)
paddings = torch.cat(paddings)

test_mpe /= len(test_loader)

print(f"Test Mean PE: {test_mpe:.4f}")



In [ ]:
from landmarker.metrics import sdr

sdr_test = sdr([2.0, 2.5, 3.0, 4.0], true_landmarks=true_landmarks, pred_landmarks=pred_landmarks,
               dim=dim_image, dim_orig=dim_origs.int(), pixel_spacing=pixel_spacings, padding=paddings)
for key in sdr_test:
    print(f"SDR for {key}mm: {sdr_test[key]:.4f}")

In [ ]:

model.eval()
model.to("cpu")
prediction_inspect_plot_transposed(ds_test2, model, [0,1]) #ds_test2.indices[:3])



In [ ]:






inspection_plot_numbers(ds_test2, range(1))



In [ ]:
from landmarker.visualize import detection_report

detection_report(true_landmarks, pred_landmarks, dim=dim_image, dim_orig=dim_origs.int(),
                    pixel_spacing=pixel_spacings, padding=paddings, class_names=ds_train.class_names,
                    radius=[2.0, 2.5, 3.0, 4.0], digits=2)



In [ ]:


from landmarker.visualize import plot_cpe

plot_cpe(true_landmarks, pred_landmarks, dim=dim_image, dim_orig=dim_origs.int(),
                    pixel_spacing=pixel_spacings, padding=paddings, class_names=ds_train.class_names,
                    group=False, title="CPE curve", save_path=None,
                    stat='proportion', unit='mm', kind='ecdf')